# LLM을 이용한 그래프 디비 설계

자연어 문서를 입력으로 받아 LLM을 이용해 Neo4j 그래프 데이터베이스의 기본 설계를 생한다.

- 문서에 등장하는 기업, 인물, 서비스, 사건, 기술, 매출 정보 등  내용을 분석하여 그래프 DB에서 사용할 노드(Label), 관계(Relationship), 속성(Property)을 도출하고, 이를 바탕으로 Neo4j에 적재할 수 있는 Cypher 쿼리를 생성한다.
- 쿼리 뿐 아니라 설계 스키마와 설계 근거를 설명하는 명세서도 같이 작성하도록 한다.

- **LLM이 생성한 스키마와 쿼리는 초안으로 사용해야 한다.** 이 초안을 바탕으로 데이터 중복, 관계 방향, Label 명명 규칙, 고유 식별자, 검색 목적 등을 사람이 검토하고 도메인에 맞게 정리한 뒤 사용하는 것이 중요하다.


In [4]:
DOCUMENTS = [
    "구글은 1995년 스탠퍼드대에서 만난 래리 페이지와 세르게이 브린이 웹페이지의 링크 구조로 중요도를 판단하는 검색엔진 BackRub을 만들면서 시작됐다. 1998년 Sun 공동창업자 앤디 벡톨샤임의 10만 달러 투자 이후 Google Inc.가 공식 출범했고, 검색을 중심으로 광고, Gmail, Maps, Android, YouTube, Cloud, AI로 확장했다. 현재는 알파벳의 핵심 자회사로, 2025년 알파벳 매출은 4,028억 달러였고 Google Services와 Google Cloud가 성장을 이끌었다.",

    "네이버는 이해진이 삼성SDS 사내벤처에서 출발해 1999년 6월 설립한 한국 인터넷 기업이다. 처음에는 검색 포털 네이버와 어린이 포털 쥬니버로 시작했고, 2000년 통합검색, 2001년 검색광고, 지식iN, 블로그, 카페, 웹툰, 쇼핑, 페이, 클라우드, AI로 영역을 넓혔다. 현재는 한국 대표 플랫폼 기업으로 검색·광고·커머스·핀테크·콘텐츠·클라우드가 주력이다. 2025년 연매출은 12조 350억 원, 영업이익은 2조 2,081억 원이었다.",

    "현재의 카카오는 두 다음과 카카오가 합쳐진 회사다. 다음은 1995년 설립되어 한메일, 다음카페, 미디어다음, 검색으로 성장했고, 카카오는 2006년 아이위랩으로 설립된 뒤 2010년 카카오톡을 내놓으며 모바일 메신저 플랫폼으로 커졌다. 2014년 다음과 카카오는 합병을 발표했고, 이후 회사명은 카카오가 되었다. 현재 주력은 카카오톡 기반 광고·커머스, 모빌리티, 페이, 콘텐츠, 음악, 미디어, AI이다. 2025년 매출은 8조 991억 원, 영업이익은 7,320억 원으로 창사 이래 최대였다.",

    "아마존은 제프 베이조스가 1994년 미국 워싱턴주 벨뷰의 차고에서 시작한 회사다. 1995년 온라인 서점 Amazon.com으로 문을 열었고, 이후 책을 넘어 전자상거래 전반, 마켓플레이스, 물류, Prime, Kindle, 영상 스트리밍, 광고로 확장했다. 2006년 시작한 AWS는 클라우드 컴퓨팅 시장의 핵심 축이 되었다. 현재 아마존은 전자상거래, 클라우드, 광고, 구독, 물류를 모두 가진 세계적 기술·유통 기업이다. 2025년 4분기 순매출은 2,134억 달러였고 AWS 매출은 356억 달러였다.",

    "엔비디아는 젠슨 황, 크리스 말라코스키, 커티스 프리엠이 1993년 창업한 반도체 기업이다. 처음에는 PC 그래픽과 게임용 GPU에 집중했고, 1999년 GeForce 256, 2006년 CUDA를 통해 GPU를 범용 병렬계산과 AI 학습에 쓰는 길을 열었다. 이후 데이터센터, AI 가속기, 네트워킹, 자율주행, 로보틱스, Omniverse 등으로 확장했다. 현재는 생성형 AI와 데이터센터 인프라의 핵심 기업이다. 2026년 1분기 회계연도 2027 매출은 816억 달러, 데이터센터 매출은 752억 달러였다.",
    
    "메타는 마크 저커버그가 더스틴 모스코비츠, 크리스 휴스, 에두아르도 세버린과 함께 2004년 2월 4일 하버드에서 Facebook을 시작하며 출발했다. 처음에는 대학생용 소셜 네트워크였지만 빠르게 일반 대중으로 확장했고, 이후 Instagram, WhatsApp, Oculus를 인수했다. 2021년에는 사명을 Meta로 바꾸고 소셜미디어와 광고, 메신저, 메타버스, AI, AR·VR 기기로 사업을 넓혔다. 현재 주력은 Facebook, Instagram, Messenger, WhatsApp을 포함한 Family of Apps 광고 사업이다. 2025년 총매출은 2,009억 달러였다."
]

## 생성 단위

1. **쿼리와 명세서를 한번에 생성한다.**
   - 문서의 내용이 크지 않을 경우 한번에 작성하도록 한다.
2. **단계적으로 나눠서 처리한다.**
   - 문서의 내용이 많을 경우 하나의 거대한 프롬프트보다 단계적으로 작성할 때 품질이 훨씬 높다.
   1. 노드와 관계 추출 및 CREATE
   2. 제약 조건 설계
   3. INDEX 설계
   4. 명세서 작성
   - 각 단계를 Prompt -> Model -> outputparser의 체인으로 구성한 뒤 연결한다.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

graph_schema_prompt = ChatPromptTemplate.from_template(
    template="""
당신은 Neo4j 그래프 데이터베이스 스키마 설계 전문가이다.

사용자가 제공한 문서를 분석하여 Neo4j에 적합한 그래프 스키마를 설계하고,
그 스키마를 기반으로 Cypher 쿼리를 작성한다.

# 목표

문서의 내용을 바탕으로 다음 작업을 수행한다.

1. 문서에서 주요 개체를 추출하여 노드(Label)를 정의한다.
2. 개체 간 의미 있는 관계를 추출하여 관계 타입(Relationship Type)을 정의한다.
3. 각 노드와 관계에 필요한 속성(Property)을 정의한다.
4. 문서의 내용을 저장하기 위한 Cypher CREATE 쿼리를 작성한다.
5. 각 노드와 관계에 대한 스키마를 표 형태로 출력한다.
6. 데이터 중복 방지와 검색 성능 향상을 위한 제약조건과 인덱스 생성 쿼리를 작성한다.

# 입력 문서

아래 문서를 분석한다.

<문서>
{document}
</문서>

# 작업 지침
- GraphRAG용 스키마를 설계하는 것으로  특히 "LLM이 생성하는 Cypher 쿼리로 탐색하기 쉬운 구조" 여야 한다.

## 1. 노드(Noe) 설계 규칙

- 문서에서 독립적으로 식별 가능한 주요 개체를 노드로 설계한다.
- 다른 개체에서 독립적으로 참조될 수 있어야 한다.
- 그 자체가 직접 검색하거나 탐색하는데 시작점이 될 수있는 개체여야 한다. 
- 사람, 조직, 장소, 상품, 문서, 사건, 개념, 기술, 프로젝트 등은 노드 후보가 될 수 있다.
- 노드 Label은 PascalCase로 작성한다.
  - 예: Person, Company, Product, Project, Document
- 하나의 노드는 최소 하나 이상의 식별 속성을 가져야 한다.
  - 예: name, title, id, code
- 문서에 명시되지 않은 정보는 임의로 만들지 않는다.
- 같은 의미의 개체는 하나의 Label로 통합한다.
  - 예: 기업, 회사, 업체 → Company

## 2. 관계(Relationship) 설계 규칙

- 노드 간 의미 있는 연결을 관계로 설계한다.
- 두 노드 사이에 방향성 있는 어떤 행위, 역할, 연관이어야 한다.
- 관계 타입은 동사 또는 동사구로 작성한다. 
- 모호한 이름은 사용하지 않는다.
- 관계 타입은 대문자와 언더스코어를 사용한다. (대문자 SNAKE_CASE)
  - 예: WORKS_FOR, LOCATED_IN, PRODUCES, PARTICIPATED_IN, BELONGS_TO
- 관계는 방향성을 가지며, 의미상 자연스러운 방향으로 설정한다.
  - 예: (Person)-[:WORKS_FOR]->(Company)
- 관계 자체에 의미 있는 정보가 있으면 관계 속성으로 저장한다.
  - 예: since, role, amount, 
- 단순히 함께 등장했다는 이유만으로 관계를 만들지 않는다.

## 3. 속성(Property) 설계 규칙
- 노드나 관계에 부가적인 정보가 있을 경우 속성으로 설정한다.
- 속성명은 camelCase로 작성한다.
  - 예: name, birthDate, createdAt, totalAmount
- 속성값은 문서에 있는 내용만 사용한다.
- 날짜, 금액, 수량, 상태, 역할 등은 가능한 속성으로 분리한다.
- 긴 설명문은 description, summary, content 등의 속성으로 저장할 수 있다.
- 배열이 필요한 경우 리스트 속성을 사용할 수 있다.
  - 예: tags: ["AI", "GraphDB"]
- 속성 타입 규칙
  - 문자열: String
  - 정수: Integer  
  - 실수: Float
  - 불리언: Boolean
  - 날짜: Date (ISO 형식 문자열)
  - 목록: List

## 4. Cypher CREATE 쿼리 작성 규칙

- 문서에서 추출한 실제 데이터를 기반으로 `CREATE` 쿼리를 작성한다.
- 노드와 관계를 함께 생성하는 Cypher 쿼리를 작성한다.
- 변수명은 소문자로 작성한다.
  - 예: person1, company1, project1
- 문자열 값은 작은따옴표 또는 큰따옴표 중 하나로 일관되게 사용한다.
- 문서에 없는 값은 생성하지 않는다.
- 동일한 노드가 여러 관계에서 사용될 경우 하나의 변수로 재사용한다.
- 여러 개의 독립적인 데이터가 있으면 가독성을 위해 여러 개의 CREATE 문으로 나누어 작성할 수 있다.

## 5. 제약조건 작성 규칙

- 각 주요 노드에는 가능한 경우 고유 식별 속성에 대해 UNIQUE 제약조건을 제안한다.
- 제약조건 이름은 명확하게 작성한다.
  - 예: person_name_unique, company_name_unique
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE CONSTRAINT person_name_unique IF NOT EXISTS
FOR (p:Person)
REQUIRE p.name IS UNIQUE;

## 6. 인덱스 작성 규칙

- 자주 검색될 가능성이 높은 속성에 대해 인덱스를 제안한다.
- 이름, 제목, 코드, 날짜, 상태, 카테고리 등은 인덱스 후보가 될 수 있다.
- UNIQUE 제약조건이 걸린 속성에는 별도 일반 인덱스를 중복으로 만들지 않는다.
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE INDEX movie_title_index IF NOT EXISTS
FOR (m:Movie)
ON (m.title);

## 7. 출력 형식

출력은 markdown으로 작성하며 반드시 아래 내용으로 구성한다.

<출력 구조>
# 1. 추출된 그래프 스키마 요약

문서에서 어떤 기준으로 노드와 관계를 설계 했는지 간단히 설명한다.

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Person | 사람 | name, role, birthDate | name |

---

# 3. 관계 스키마 예

| 관계 타입 | 시작 노드 | 종료 노드 | 설명 | 관계 속성 |
|---|---|---|---|---|
| WORKS_FOR | Person | Company | 사람이 회사에 소속됨 | role, since |

---

# 4. Cypher CREATE 쿼리

```cypher
// 문서 내용을 기반으로 노드와 관계를 생성하는 쿼리
```

# 5. 제약조건 생성 쿼리
```cypher
// UNIQUE 제약조건 생성 쿼리
```

# 6. Index 생성 쿼리
```cypher
// 검색 성능 향상을 위한 인덱스 생성 쿼리
```

7. 설계 근거

* 왜 해당 개체를 노드로 설계했는지 설명한다.
* 왜 해당 연결을 관계로 설계했는지 설명한다.
* 어떤 속성을 식별자 또는 검색 대상으로 판단했는지 설명한다.
* 문서에 정보가 부족하여 확정하기 어려운 부분이 있다면 명시한다.
</출력 구조>
---

## 8. 주의사항

- 문서에 없는 내용을 추측해서 만들지 않는다.
- Neo4j에서 실행 가능한 Cypher 문법으로 작성한다.
- Label, Relationship Type, Property 이름은 일관된 네이밍 규칙을 따른다.
- 가능한 한 단순하고 명확한 그래프 구조를 우선한다.
- 지나치게 많은 노드와 관계를 만들지 말고, 실제 질의에 유용한 구조를 설계한다.
""")

In [6]:
##########################################
# 지식 그래프로 변환
##########################################
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


model = ChatOpenAI(model="gpt-5.5")

chain = graph_schema_prompt | model

def convert_to_graph_doc(docs:list[str], labels: list[str]=None, relationship:list[str]=None) -> list[str]:
    """문서들을 받아서 db schema(index, constraint) 와 data 생성(Node, Relationship) 생성 cypher query 만드는 함수

    Args:
        docs (list[str]): 쿼리를 만들 문서들
        labels (list[str], optional): 추출할 Node의 Label들. Defaults to None.
        relationship (list[str], optional): 추출할 관계들. Defaults to None.

    Returns:
        list[str]: 문서별로 생성된 결과를 리스트에 담아서 반환한다.
    """
    graph_docs = []
    for doc in docs:
        result = chain.invoke(
                {
                    "labels": labels if labels else "None",
                    "relationship": relationship if relationship else "None",
                    "document": doc
                }
        )
        graph_docs.append(result)
    
    return graph_docs

In [8]:
graph_docs = convert_to_graph_doc(DOCUMENTS)

In [12]:
from IPython.display import Markdown

# Markdown(graph_docs[5].content)
print(graph_docs[5].content)

# 1. 추출된 그래프 스키마 요약

문서의 핵심은 **Meta의 기원, 창업자, 시작 장소, 인수한 서비스/제품, 사업 확장 영역, 현재 주력 사업, 2025년 매출**이다.  
따라서 GraphRAG에서 탐색하기 쉽도록 다음 기준으로 스키마를 설계했다.

- **Meta**는 독립적으로 조회되는 중심 개체이므로 `Company` 노드로 설계
- **마크 저커버그 등 창업자**는 `Person` 노드로 설계
- **Facebook, Instagram, WhatsApp, Oculus, Messenger**는 회사가 운영하거나 인수한 주요 서비스/제품이므로 `Product` 노드로 설계
- **하버드**는 Facebook이 시작된 장소이므로 `Location` 노드로 설계
- **소셜미디어, 광고, 메신저, 메타버스, AI, AR·VR 기기**는 Meta의 사업 영역이므로 `BusinessArea` 노드로 설계
- **Family of Apps 광고 사업**은 현재 주력 사업 단위이므로 `BusinessSegment` 노드로 설계
- **2025년 총매출 2,009억 달러**는 연도와 금액이 있는 재무 정보이므로 `FinancialMetric` 노드로 설계

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Company | 기업 | name, aliases, formerName, renamedYear | name |
| Person | 사람 | name | name |
| Product | 서비스, 앱, 제품 | name, initialAudience, expandedAudience | name |
| Location | 장소 | name | name |
| BusinessArea | 사업 영역 | name | name |
| BusinessSegment | 사업 부문 | name, description | name |
| FinancialMetric | 재무 지표 | id, m

# graph_docs를 실제 Neo4j에 로딩

위에서 `convert_to_graph_doc(DOCUMENTS)`로 만든 `graph_docs`는 아직 텍스트(마크다운 + Cypher 코드블록)일 뿐,
실제 Neo4j 데이터베이스에는 적재되지 않은 상태다.

여기서는 `graph_docs`의 Cypher CREATE 쿼리를 **`Node`/`Relationship` 객체로 파싱**한 뒤,
`add_graph_documents()`로 Neo4j에 저장한다.

**이 방식을 쓰는 이유**: `add_graph_documents()`는 내부적으로 `MERGE`를 사용한다.
즉 같은 `id`+`type`을 가진 노드가 여러 번 들어와도 자동으로 하나로 합쳐진다.
문서별로(구글/네이버/카카오/...) 독립적으로 생성된 Cypher를 그대로 `CREATE`로 실행하면
여러 회사가 공통으로 가진 개체(예: `AI`라는 `BusinessArea`)가 문서마다 중복 생성되거나
`UNIQUE` 제약조건과 충돌할 수 있는데, `node_dict`를 이용한 아래 방식은 이런 중복을
파싱 단계에서부터 방지한다.


In [ ]:
# !uv pip install langchain_neo4j neo4j
# 설치 후 커널 재시작


In [ ]:
# docker run \
#     --name neo4j \
#     -p 7474:7474 -p 7687:7687 \
#     -d \
#     -e NEO4J_AUTH=neo4j/password \
#     -e NEO4J_PLUGINS='["apoc"]' \
#     neo4j:latest
# 브라우저 콘솔: http://localhost:7474 (neo4j / password 로 로그인)


## 1. Neo4j 연결

`.env` 파일에 아래 항목을 추가해두고 사용한다.

```
NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=password
NEO4J_DATABASE=neo4j
```


In [ ]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE", "neo4j"),
)


## 2. `graph_docs` → Cypher 텍스트에서 Node/Relationship 추출

`graph_docs[i].content`(마크다운) 안의 ` ```cypher ... ``` ` 코드블록에서
노드 선언(`(var:Label {props})`)과 관계(`(var)-[:TYPE {props}]->(var)`) 패턴을 정규식으로 뽑아
`Node`/`Relationship` 객체로 변환한다.


In [ ]:
import re
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship


# ── 마크다운에서 cypher 코드블록 추출 ──
def extract_cypher_blocks(markdown_text: str) -> "list[str]":
    return re.findall(r"```cypher\s*(.*?)```", markdown_text, flags=re.DOTALL)


def extract_data_blocks(markdown_text: str) -> "list[str]":
    """CREATE 데이터 블록만 남기고, CONSTRAINT/INDEX 전용 블록은 제외한다.
    (CONSTRAINT 구문의 `FOR (c:Company)` 부분이 노드 선언 패턴과 겹쳐
    빈 유령 노드로 잘못 파싱되는 것을 막기 위함)
    """
    blocks = extract_cypher_blocks(markdown_text)
    return [b for b in blocks if not re.search(r"CREATE\s+(CONSTRAINT|INDEX)", b, re.IGNORECASE)]


# ── Cypher 속성 맵({key: value, ...}) 파싱 ──
def _split_top_level(s: str, sep: str = ",") -> "list[str]":
    """괄호/따옴표 안의 콤마는 무시하고 최상위 콤마 기준으로만 분리한다."""
    parts, depth, buf, quote = [], 0, "", None
    for ch in s:
        if quote:
            buf += ch
            if ch == quote:
                quote = None
        elif ch in "\'\"":
            quote = ch
            buf += ch
        elif ch in "([{":
            depth += 1
            buf += ch
        elif ch in ")]}":
            depth -= 1
            buf += ch
        elif ch == sep and depth == 0:
            parts.append(buf)
            buf = ""
        else:
            buf += ch
    if buf.strip():
        parts.append(buf)
    return parts


def _parse_value(v: str):
    v = v.strip()
    m = re.match(r"^\w+\(\s*([\'\"])(.*?)\1\s*\)$", v)  # date(\'2004-02-04\') 같은 함수 호출
    if m:
        return m.group(2)
    if v.startswith("[") and v.endswith("]"):
        return [_parse_value(x) for x in _split_top_level(v[1:-1])]
    if len(v) >= 2 and v[0] == v[-1] and v[0] in "\'\"":
        return v[1:-1]
    if v.lower() in ("true", "false"):
        return v.lower() == "true"
    try:
        return int(v)
    except ValueError:
        try:
            return float(v)
        except ValueError:
            return v


def parse_properties(props_block: str) -> dict:
    props_block = (props_block or "").strip()
    if not (props_block.startswith("{") and props_block.endswith("}")):
        return {}
    props = {}
    for part in _split_top_level(props_block[1:-1]):
        if ":" not in part:
            continue
        key, _, val = part.partition(":")
        props[key.strip()] = _parse_value(val)
    return props


# ── Cypher CREATE 블록 -> Node/Relationship 객체로 파싱 ──
NODE_PATTERN = re.compile(r"\(\s*(\w+)\s*:\s*(\w+)\s*(\{.*?\})?\s*\)")
REL_PATTERN = re.compile(
    r"\(\s*(\w+)\s*\)\s*-\s*\[\s*:\s*(\w+)\s*(\{.*?\})?\s*\]\s*->\s*\(\s*(\w+)\s*\)"
)
IDENTIFYING_KEYS = ["name", "id", "title", "code"]  # 노드를 식별하는 우선순위

node_dict = {}       # (type, 식별값) -> Node   ※ 여러 문서에 걸쳐 자동 dedup됨
relationships = []

for doc in graph_docs:
    content = doc.content if hasattr(doc, "content") else str(doc)
    var_to_key = {}  # 이 문서 안에서만 유효한 변수명 -> node_dict 키 매핑

    for block in extract_data_blocks(content):
        # 노드 선언: (var:Label {props})
        for var, label, props_str in NODE_PATTERN.findall(block):
            props = parse_properties(props_str)
            id_key = next((props[k] for k in IDENTIFYING_KEYS if k in props), var)
            key = (label, str(id_key))

            if key in node_dict:
                node_dict[key].properties.update(props)  # 겹치는 노드는 속성만 병합
            else:
                node_dict[key] = Node(id=str(id_key), type=label, properties=props)

            var_to_key[var] = key

        # 관계: (var)-[:TYPE {props}]->(var)
        for src_var, rel_type, props_str, tgt_var in REL_PATTERN.findall(block):
            src_key, tgt_key = var_to_key.get(src_var), var_to_key.get(tgt_var)
            if src_key is None or tgt_key is None:
                continue  # 노드 선언을 못 찾은 관계는 건너뜀
            relationships.append(
                Relationship(
                    source=node_dict[src_key],
                    target=node_dict[tgt_key],
                    type=rel_type,
                    properties=parse_properties(props_str),
                )
            )

nodes = list(node_dict.values())
print(f"노드 {len(nodes)}개, 관계 {len(relationships)}개 파싱 완료")


## 3. GraphDocument 생성 후 Neo4j에 저장


In [ ]:
# GraphDocument 객체 생성
graph_doc = GraphDocument(
    nodes=nodes,
    relationships=relationships,
    source=Document(page_content="\n\n".join(DOCUMENTS)),  # GraphDocument는 source가 필수 항목
)

# 생성된 GraphDocument를 Neo4j 데이터베이스에 저장
# add_graph_documents 메서드는 GraphDocument 객체 리스트를 받아 데이터베이스에 일괄 저장
graph.add_graph_documents([graph_doc])  # Node, Relationship들을 MERGE 처리.
graph.refresh_schema()
print("그래프 데이터베이스에 저장 완료")


## 4. 적재 결과 확인


In [ ]:
print(graph.schema)


In [ ]:
# 노드/관계 개수 확인
counts = graph.query(
    "MATCH (n) RETURN count(n) AS node_count"
)[0]["node_count"]
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN count(r) AS rel_count"
)[0]["rel_count"]

print(f"저장된 노드 수: {counts}")
print(f"저장된 관계 수: {rel_counts}")
